In [1]:
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


### ESTUDIANTES ACTIVOS

In [2]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT * FROM Estudiantes_activos
            WHERE COD_PERIODO NOT LIKE '%I%' AND COD_PERIODO NOT LIKE '%E%'
            
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_activos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [3]:
df_activos['NUM_IDENTIFICACION'] = df_activos['NUM_IDENTIFICACION'] .astype(str)

Se realiza cruce con tabla que se genera despues de correr script de python para grados

In [4]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
        SELECT DISTINCT
            a.Identificacion,
            a.Nombre_Estudiante,
            a.Periodo,
            a.Cod_Unidad_sug AS Cod_Unidad,
            a.Status,
            a.Nuevo,
            a.promedio,
            a.Semestre_SINU,
            a.ciclo,
            a.SEMESTRE_MEN,
            a.Año,
            a.AÑO_MEN,
            a.[Estado_Alumno],
            a.anio_grado,
            a.Regional,
            a.Programa,
            a.Grado_TC, 
            a.Grado_TG, 
            a.Grado_PR,
            a.Pensum_limpio AS pensum,
            c.SNIES,
            c.ESTADO_IDIOMAS,
            c.ESTADO_ACADEMICO,
            c.ESTADO_DEUDA,
            c.ESTADO_GRADO
        FROM
            DESERCION.Python_CUN_Simulado_cambios_ciclo a
        LEFT JOIN
            grados.Grados_Saldo c
        ON
                a.Identificacion = c.NUM_IDENTIFICACION
            --AND a.Cod_Unidad = c.COD_UNIDAD
            AND a.Pensum = c.COD_PENSUM
            AND a.ciclo = c.ciclo
        
        ORDER BY
            a.Identificacion,
            a.Periodo;
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [5]:
df = df[df['Identificacion'].isin(df_activos['NUM_IDENTIFICACION'])]

### Se realiza el conteo del numero de repeticiones de cada registro, sin tener en cuenta estado retiro y desercion CUN

In [6]:
df_n = df[(df['Status'] != 'Desercion CUN') & (df['Status'] != 'Retiro')]
df['num_repeticiones'] = df_n.groupby(['ciclo', 'Identificacion']).cumcount() + 1

### Se realiza el calculo de primer y ultimo curso en dos columnas nuevas del df

In [7]:
df['Primer_curso'] = df.groupby(['Identificacion', 'ciclo']).cumcount() == 0
df['Primer_curso'] = df['Primer_curso'].replace({True: 'SI', False: 'NO'})

df['ultimo_curso'] = df.groupby(['Identificacion', 'ciclo']).cumcount() == df.groupby(['Identificacion', 'ciclo'])['ciclo'].transform('count') - 1
df['ultimo_curso'] = df['ultimo_curso'].replace({True: 'SI', False: 'NO'})

#Se filtra del df para no tener en cuenta los grupos de registros cuyo estado es igual a graduado en cada ciclo

In [8]:
df.loc[(df['ciclo']=='TECNICO')&(df['Grado_TC']!='-'),'Estado_Graduado'] = 'Graduado'
df.loc[(df['ciclo']=='TECNOLOGO')&(df['Grado_TG']!='-'),'Estado_Graduado'] = 'Graduado'
df.loc[(df['ciclo']=='PROFESIONAL')&(df['Grado_PR']!='-'),'Estado_Graduado'] = 'Graduado'

In [9]:
# Crear una marca booleana de si ya está graduado o no
df['Graduado_Marca'] = df['Estado_Graduado'] == 'Graduado'

# Agrupar por estudiante y ciclo, y verificar si alguno está Graduado
df['Graduado_Final'] = df.groupby(['Identificacion', 'ciclo'])['Graduado_Marca'].transform('max')

# Reemplazar True/False por 'Graduado' o mantener NaN
df['Estado_Graduado'] = df['Graduado_Final'].apply(lambda x: 'Graduado' if x else None)

# Eliminar columna temporal si no la necesitas
df.drop(columns=['Graduado_Marca', 'Graduado_Final'], inplace=True)


In [10]:
# # Filtra las filas donde alguna tiene 'Graduado' en la columna 'Status'
df= df[~df.groupby(['Identificacion', 'ciclo'])['Estado_Graduado'].transform(lambda x: 'Graduado' in x.values)]
# df[['Identificacion', 'Status', 'ultimo_curso' ,'Año','SEMESTRE_MEN','ciclo' ,'num_repeticiones']]

#Funcion para evaluar ciclo por semestres cursados

In [11]:
# Función para evaluar las otras condiciones y asignar valores a 'Grado_oportuno'
def evaluar_condiciones(row):
    ciclo = row['ciclo']
    if ciclo == 'TECNICO' and 4 <= row['num_repeticiones'] <= 6:
        return 'Grado Oportuno Tecnico'
    elif ciclo == 'TECNOLOGO' and 3 <= row['num_repeticiones'] <= 5:
        return 'Grado Oportuno Tecnologo'
    elif ciclo == 'PROFESIONAL' and 3 <= row['num_repeticiones'] <= 5:
        return 'Grado Oportuno Profesional'

    return 'No aplica'

# Aplicar la función a cada fila del DataFrame y crear una nueva columna 'Grado_oportuno'
df['Grado_oportuno'] = df.apply(evaluar_condiciones, axis=1)

In [12]:
df['Año'] = df['Año'].replace('', 0).astype(int)
df['SEMESTRE_MEN'] = df['SEMESTRE_MEN'].replace('', 0).astype(int)



In [13]:
from datetime import datetime

# Obtener el año y semestre actual
hoy = datetime.today()
año_actual = hoy.year
semestre_actual = 1 if hoy.month <= 6 else 2  # Determinar el semestre actual

In [14]:
# Crear la columna con el año y semestre actual en el mismo formato de 'añoMen'
df['Año_Semestre_Actual'] = df.apply(lambda x: f"{año_actual}-{semestre_actual}", axis=1)

# Calcular la diferencia en semestres
df['Diferencia_Semestres'] = (año_actual - df['Año']) * 2 + (semestre_actual - df['SEMESTRE_MEN'])

In [15]:
# Asignar 'No aplica' si la diferencia de semestres es mayor a 1 para TECNICO, TECNOLOGO o PROFESIONAL
df.loc[
    (df['ciclo'].isin(['TECNICO', 'TECNOLOGO', 'PROFESIONAL'])) & (df['Diferencia_Semestres'] > 1),
    'Grado_oportuno'
] = 'No aplica'

#Se realiza el conteo del num de repeticiones para el registro grado oportuno

In [16]:
#Conteo de num repeticiones grado oportuno
df['conteo_grado_oportuno'] = df[df['Grado_oportuno'] != 'No aplica'].groupby(['Grado_oportuno', 'Identificacion']).cumcount() + 1
df['conteo_grado_oportuno'].fillna(0, inplace=True)

# Convertir la columna a tipo entero si es necesario
df['conteo_grado_oportuno'] = df['conteo_grado_oportuno'].astype(int)
#df[['Identificacion', 'Periodo', 'AÑO_MEN','ciclo', 'Status', 'ultimo_curso','Primer_curso', 'num_repeticiones','Grado_oportuno', 'conteo_grado_oportuno' ]]

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20688\162960369.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['conteo_grado_oportuno'].fillna(0, inplace=True)


Obtener el último registro para cada cédula

In [17]:
df['last_val_grad_oport'] = df.groupby('Identificacion')['conteo_grado_oportuno'].transform('last')
df['last_val_semestre'] = df.groupby('Identificacion')['SEMESTRE_MEN'].transform('last')

df['last_val_semestre'] = df['last_val_semestre'].astype(int)
df['Año'] = df['Año'].astype(int)
df['SEMESTRE_MEN'] = df['SEMESTRE_MEN'].astype(int)

## Se filtra el df para que no me traiga los registros donde no aplica grado oportuno -- Aqui se filra o no, dependiendo si piden la marca de grado oportuno o el historico

In [18]:
df_no_aplica = df[df['Grado_oportuno']=='No aplica']

In [19]:
df_aplica = df[df['Grado_oportuno'] != 'No aplica']

## Esta parte solo se ejecuta cuando necesitanos historico desde el 2022

## -------------------------------------

In [20]:
#df['AÑO_SEM_GRADO'] = df['grad_annio'].astype(str) + '-' + df['grad_semestre'].astype(str)
##FUNCION QUE TOMA EL VALOR DE LAS COLUMNAS Grado_TC, Grado_TG, Grado_PR y los pone en la columna AÑO_SEM_GRADO

def determinar_grado(row):
    if pd.notna(row['Grado_TC']):
        return row['Grado_TC']
    elif pd.notna(row['Grado_TG']):
        return row['Grado_TG']
    elif pd.notna(row['Grado_PR']):
        return row['Grado_PR']
    else:
        return 'NO_aplica'

# Aplicar la función al DataFrame para crear la nueva columna
df_aplica['AÑO_SEM_GRADO'] = df_aplica.apply(determinar_grado, axis=1)


C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20688\4156019923.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aplica['AÑO_SEM_GRADO'] = df_aplica.apply(determinar_grado, axis=1)


Funcion que me evalua y crea los nuevos registros a partir del ultimo dependiendo el numero de veces que esta el registro 

In [21]:
from tqdm import tqdm

def add_semesters(current_year, current_semester, n_semesters_to_add):
    total_semesters = current_semester + n_semesters_to_add
    new_year = current_year + total_semesters // 2
    new_semester = total_semesters % 2 + 1
    return new_year, new_semester

#Agrupa por identificacion y ciclo, y rankea
cols_g = ['Identificacion', 'ciclo']
df_aplica['n'] = df_aplica.groupby(cols_g)['conteo_grado_oportuno'].rank(ascending=False)
#Obtener el mayor 
df_max = df_aplica[df_aplica['n'] == 1]

ids = df_aplica['Identificacion'].drop_duplicates()
df_final = df_aplica[['Identificacion', 'Periodo','Estado_Alumno', 'Año', 'SEMESTRE_MEN', 'ciclo', 'Grado_oportuno','Programa','Regional', 
               'ESTADO_IDIOMAS','ESTADO_ACADEMICO', 'ESTADO_DEUDA', 'ESTADO_GRADO', 'AÑO_MEN', 'AÑO_SEM_GRADO','pensum','Cod_Unidad','Semestre_SINU']]

data = []
for id in tqdm(ids):
  df_id = df_max[df_max['Identificacion'] == id]
  for ciclo in df_id['ciclo'].drop_duplicates():
    df_id_ciclo = df_id[df_id['ciclo'] == ciclo]
    n = 4 - df_id_ciclo['conteo_grado_oportuno'].values[0]
    
    for i in range(n):
      year, periodo, semester, grado_oportuno, programa, regional, idiomas, academico, deuda, est_grado,anio_men, año_sem_grad, pensum, cod_unidad,semestre_sinu = (df_id_ciclo['Año'].values[0], 
      df_id_ciclo['Periodo'].values[0], df_id_ciclo['SEMESTRE_MEN'].values[0],df_id_ciclo['Grado_oportuno'].values[0], 
      df_id_ciclo['Programa'].values[0], df_id_ciclo['Regional'].values[0], df_id_ciclo['ESTADO_IDIOMAS'].values[0], df_id_ciclo['ESTADO_ACADEMICO'].values[0],
      df_id_ciclo['ESTADO_DEUDA'].values[0], df_id_ciclo['ESTADO_GRADO'].values[0], df_id_ciclo['AÑO_MEN'].values[0],
      df_id_ciclo['AÑO_SEM_GRADO'].values[0],df_id_ciclo['pensum'].values[0],df_id_ciclo['Cod_Unidad'].values[0],df_id_ciclo['Semestre_SINU'].values[0] )
      
      #Evaluacion de la funcion 
      n_year, n_semester = add_semesters(year, semester, i)
      data.append([id, periodo, 'Grado Oportuno', n_year, n_semester, ciclo, grado_oportuno, programa, regional,idiomas, academico, deuda, est_grado, anio_men, año_sem_grad,pensum,cod_unidad,semestre_sinu])

df_f = pd.concat([df_final, pd.DataFrame(data, columns = df_final.columns)])


C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20688\4252945662.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_aplica['n'] = df_aplica.groupby(cols_g)['conteo_grado_oportuno'].rank(ascending=False)
100%|██████████| 13638/13638 [00:50<00:00, 269.39it/s]


Ordenar los registros del df

In [22]:
df_f = df_f.sort_values(by=['Identificacion','Grado_oportuno', 'ciclo'])

#Se realiza conteo de la columna Grado_Oportuno

In [23]:
df_f['Cumcount_grad'] = df_f.groupby(['Identificacion','Grado_oportuno', 'ciclo']).cumcount() + 1

Evaluacion de las condiciones para si cumple academicamente, ingles y esta al dia con pagos

In [24]:
import numpy as np
# Condiciones y valores a asignar
condiciones = [
    (df_f['Grado_oportuno'] == 'Grado Oportuno Profesional') &
    (df_f['ESTADO_IDIOMAS'].isin(['CUMPLE', 'INGLES INCLUIDO EN MALLA'])) &
    (df_f['ESTADO_ACADEMICO'].isin(['CUMPLE', 'Homologado por Reconocimiento de Titulo'])) &
    (df_f['ESTADO_DEUDA'] == 'CUMPLE'),  #&
    #  (df_f['PENSAMIENTO_CUNISTA'] == 'Aprueba') &
    #   (df_f['PRACTICA'] == 'Aprueba'),
    
    (df_f['Grado_oportuno'] == 'Grado Oportuno Tecnico') &
    (df_f['ESTADO_IDIOMAS'].isin(['CUMPLE', 'INGLES INCLUIDO EN MALLA'])) &
    (df_f['ESTADO_ACADEMICO'].isin(['CUMPLE', 'Homologado por Reconocimiento de Titulo'])) &
    (df_f['ESTADO_DEUDA'] == 'CUMPLE'), #&
    #  (df_f['PENSAMIENTO_CUNISTA'] == 'Aprueba') &
    #   (df_f['PRACTICA'] == 'Aprueba'),
    
    (df_f['Grado_oportuno'] == 'Grado Oportuno Tecnologo') &
    (df_f['ESTADO_IDIOMAS'].isin(['CUMPLE', 'INGLES INCLUIDO EN MALLA'])) &
    (df_f['ESTADO_ACADEMICO'].isin(['CUMPLE', 'Homologado por Reconocimiento de Titulo']))&
    (df_f['ESTADO_DEUDA'] == 'CUMPLE') #&
    #  (df_f['PENSAMIENTO_CUNISTA'] == 'Aprueba') &
    #   (df_f['PRACTICA'] == 'Aprueba')
]

valores = ['Apto Grado oportuno profesional', 'Apto Grado oportuno tecnico', 'Apto Grado oportuno tecnologo']

# Aplicar la condición y asignar valores

df_f['Apto_Grado_oportuno'] = np.select(condiciones, valores, 'NO APTO')

In [25]:
df_f[['Identificacion', 'ciclo', 'Grado_oportuno', 'Estado_Alumno', 'Cumcount_grad', 'AÑO_SEM_GRADO', 'AÑO_MEN' ]].iloc[50:70]
#df_f['Cumcount_grad'].dtype
df_f['Fecha_ministerio'] = df_f['Año'].astype(str) + '-' + df_f['SEMESTRE_MEN'].astype(str)
#df_f[['Identificacion', 'Status', 'Año','SEMESTRE_MEN','ciclo' ,'Cumcount_grad', 'Grado_oportuno', 'Fecha_ministerio', 'AÑO_SEM_GRADO']]

In [26]:
import numpy as np
# Condiciones y valores a asignar
condiciones_g = [
    (df_f['Grado_oportuno'] == 'Grado Oportuno Profesional') &
    #(df_f['Status'] == 'Grado Oportuno') &
    (df_f['Cumcount_grad']== 2) &
    (df_f['Fecha_ministerio'] == df_f['AÑO_SEM_GRADO'] ),
    
    (df_f['Grado_oportuno'] == 'Grado Oportuno Tecnico') &
    #(df_f['Status'] == 'Grado Oportuno') &
    (df_f['Cumcount_grad']== 2) &
    (df_f['Fecha_ministerio'] == df_f['AÑO_SEM_GRADO'] ),
    
    (df_f['Grado_oportuno'] == 'Grado Oportuno Tecnologo') &
    #(df_f['Status'] == 'Grado Oportuno') &
    (df_f['Cumcount_grad']== 2) &
    (df_f['Fecha_ministerio'] == df_f['AÑO_SEM_GRADO'] )
]

valores = ['Tuvo graduacion oportuna profesional', 'Tuvo graduacion oportuna tecnico', 'Tuvo graduacion oportuna tecnologo']

# Aplicar la condición y asignar valores

df_f['FUE_GRADO_OPORTUNO'] = np.select(condiciones_g, valores, 'NO FUE GRADUACION OPORTUNA')
#df_f['FUE_GRADO_OPORTUNO'].unique()



AQUI VOY


In [27]:
# Aplicar cumcount() a la columna "Grado_oportuno"
#df_f['Cumcount_Grado_oportuno'] = df_f.groupby('Identificacion')['Grado_oportuno'].cumcount() + 1
df_f['Cumcount_Grado_oportuno'] = df_f.groupby(['Identificacion', 'ciclo'])['Apto_Grado_oportuno'].cumcount() + 1


In [28]:
mapeo_reemplazos = {
    1: ' - ',
    2: 'Primer Grado Oportuno',
    3: 'Segundo Grado Oportuno',
    4: 'Tercer Grado Oportuno'
    # Agrega más pares clave-valor según sea necesario
}

# Realiza el reemplazo en la columna 'Columna_a_Reemplazar'
df_f['Tipo_grado'] = df_f['Cumcount_Grado_oportuno'].replace(mapeo_reemplazos)

In [29]:
df_f= df_f[['Identificacion','Semestre_SINU','pensum','Cod_Unidad', 'Periodo', 'Estado_Alumno', 'Año', 'SEMESTRE_MEN', 'ciclo',
       'Grado_oportuno', 'Programa', 'Regional', 'ESTADO_IDIOMAS',
       'ESTADO_ACADEMICO','AÑO_MEN', 'AÑO_SEM_GRADO', 'Cumcount_grad',
       'Apto_Grado_oportuno', 'Fecha_ministerio',
       'Cumcount_Grado_oportuno', 'Tipo_grado']]

In [30]:
df_f = df_f[df_f['Tipo_grado'].isin(['Primer Grado Oportuno','Segundo Grado Oportuno'])]

In [31]:
df_f['pensum'] = df_f['pensum'].str.rstrip('ND')

In [32]:
df_f['llave'] = df_f['Identificacion'] + '_' + df_f['ciclo']
df_no_aplica['llave'] = df_no_aplica['Identificacion'] + '_' + df_no_aplica['ciclo']

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20688\2168431933.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_no_aplica['llave'] = df_no_aplica['Identificacion'] + '_' + df_no_aplica['ciclo']


In [33]:
df_diferentes = df_no_aplica[~df_no_aplica['llave'].isin(df_f['llave'])]

In [34]:
df_diferentes['Semestre_SINU'] = df_diferentes['Semestre_SINU'].astype('int64')

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_20688\502576889.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_diferentes['Semestre_SINU'] = df_diferentes['Semestre_SINU'].astype('int64')


In [35]:
df_diferentes = df_diferentes.sort_values(by=['Semestre_SINU','Periodo'])

In [36]:
# df_diferentes = df_diferentes[~df_diferentes['Status'].isin(['Desercion CUN','Retiro'])]

In [37]:
df_max_semestre = df_diferentes.groupby(['Identificacion', 'ciclo']).last().reset_index()

In [38]:
# # Obtener el semestre máximo por identificación y ciclo
# df_max_semestre = df_diferentes.loc[df_diferentes.groupby(['Identificacion', 'ciclo'])['Semestre_SINU'].idxmax()]

### OBTENER AÑO SEMESTRE ACTUAL

In [39]:
from datetime import datetime
# Obtener el año y semestre actual
hoy = datetime.today()
año_actual = hoy.year
semestre_actual = 1 if hoy.month <= 6 else 2  # Determinar el semestre actual

# Crear la columna con el año y semestre actual en el mismo formato de 'añoMen'
df_max_semestre['AÑO_MEN_ACTUAL'] = df_max_semestre.apply(lambda x: f"{año_actual}-{semestre_actual}", axis=1)

In [40]:
# Calcular la diferencia en semestres
df_max_semestre['Diferencia_Semestres'] = (año_actual - df_max_semestre['Año']) * 2 + (semestre_actual - df_max_semestre['SEMESTRE_MEN'])

In [41]:
df_max_semestre['Semestre_SINU'] = df_max_semestre['Semestre_SINU'].astype('Int32')

In [42]:
# Aplicar la condición para asignar 'Tipo_grado'
def determinar_tipo_grado(row):
    if row['ciclo'] == 'TECNICO' and row['Diferencia_Semestres'] > 1:
        return 'Fuera de su grado oportuno'
    elif row['ciclo'] == 'TECNOLOGO'  and row['Diferencia_Semestres'] > 1:
        return 'Fuera de su grado oportuno'
    elif row['ciclo'] == 'PROFESIONAL' and row['Diferencia_Semestres'] > 1:
        return 'Fuera de su grado oportuno'
    else:
        return 'Aun no se debe hacer seguimiento'

# Crear la columna Tipo_grado
df_max_semestre['Tipo_grado'] = df_max_semestre.apply(determinar_tipo_grado, axis=1)

#### CONCATEANACIÓN DE LOS QUE APLICAN PARA GRADO OPORTUNO Y LOS NO APLICAN

In [43]:
df_concat = pd.concat([df_f,df_max_semestre],ignore_index=True)

In [44]:
df_concat[df_concat['Identificacion']=='1123623554']

,Identificacion,Semestre_SINU,pensum,Cod_Unidad,Periodo,Estado_Alumno,Año,SEMESTRE_MEN,ciclo,Grado_oportuno,...,num_repeticiones,Primer_curso,ultimo_curso,Estado_Graduado,Año_Semestre_Actual,Diferencia_Semestres,conteo_grado_oportuno,last_val_grad_oport,last_val_semestre,AÑO_MEN_ACTUAL


In [45]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'GRADO_OPORTUNO_ESTUDIANTES_ACTIVOS'
schema_name = 'SRUBIO'

# Send the data to sql server
df_concat.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

#### OPCION DE GRADO - CORRER SCRIPT opcion_grado_est_unicos.ipynb

In [46]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            	SELECT DISTINCT a.Identificacion,
					a.Periodo,
					a.ciclo,
					a.Semestre_SINU,
					a.pensum,
					a.Cod_Unidad,a.Año as AÑO_GRAD_OPORTUNA,
					a.SEMESTRE_MEN as SEMESTRE_MEN_GRAD_OPORTUNA,
					a.Programa,
					a.Regional,
					a.ESTADO_ACADEMICO,
					a.ESTADO_IDIOMAS,
					a.Tipo_grado,
					b.Resultado as OPCION_GRADO

	FROM SRUBIO.GRADO_OPORTUNO_ESTUDIANTES_ACTIVOS a
	LEFT JOIN SRUBIO.OPCION_GRADO_ESTUDIANTES_ACTIVOS b
	ON CONVERT(VARCHAR,a.Identificacion) = CONVERT(VARCHAR,b.NUM_IDENTIFICACION) AND a.ciclo = b.NIVEL
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_op = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [47]:
df_op.loc[(df_op['OPCION_GRADO'].isna()),'OPCION_GRADO'] = 'SIN CURSAR'

### TABLA DE NIVELES APROBADOS INGLES

##### DENTRO DE MALLA

In [48]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
                        
            
                SELECT * 
                FROM OPENQUERY([172.16.1.175], 
                    '
                    SELECT 
                        b.NUM_IDENTIFICACION, 
                        a.COD_MATERIA
                    FROM 
                        SINU.SRC_HIS_ACADEMICA a
                        LEFT JOIN SINU.SRC_ALUM_PROGRAMA c ON a.ID_ALUM_PROGRAMA = c.ID_ALUM_PROGRAMA
                        LEFT JOIN SINU.BAS_TERCERO b ON b.ID_TERCERO = c.ID_TERCERO
                    WHERE 
                        a.COD_MATERIA IN (''A1I00'', ''A2I00'', ''B1I00'', ''564I6'', ''564I7'', ''564I8'', 
                                            ''564I9'', ''564I1'', ''564I2'', ''564I3'', ''564I4'', ''564I5'', 
                                            ''56N10'', ''56N11'', ''56N12'', ''56N13'', ''A1I00'', ''56I10'') 
                        AND a.IND_APROBADA = ''1''
                        
                    '
                );
            
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_dm = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [49]:
df_dm = df_dm.drop_duplicates()

In [50]:
df_dm_conteo = df_dm.groupby('NUM_IDENTIFICACION')['COD_MATERIA'].count().reset_index()

In [51]:
df_dm_conteo['TIPO_INGLES_MALLA'] = 'INGLES INCLUIDO EN MALLA'

In [52]:
df_dm_conteo.rename(columns={'COD_MATERIA':'NIVELESAPROBADOS'},inplace=True)

#### FUERA DE MALLA

In [53]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
                        SELECT * 
            FROM OPENQUERY([172.16.1.175], 
                '
                SELECT 
                    B.NUM_IDENTIFICACION, 
                    SUM(a.IND_APROBADA) AS NivelesAprobados
                FROM 
                    SINU.SRC_HIS_ACADEMICA a
                    LEFT JOIN SINU.SRC_ALUM_PROGRAMA c ON a.ID_ALUM_PROGRAMA = c.ID_ALUM_PROGRAMA
                    LEFT JOIN SINU.BAS_TERCERO b ON b.ID_TERCERO = c.ID_TERCERO
                WHERE 
                    (a.COD_PERIODO LIKE ''%BE%'' OR a.COD_PERIODO LIKE ''%I%'' OR a.COD_PERIODO LIKE ''%M%'') 
                    AND a.IND_APROBADA = ''1''
                GROUP BY 
                    B.NUM_IDENTIFICACION
    '
)
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_ing = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [54]:
df_ing['TIPO_INGLES_MALLA'] = 'FUERA DE MALLA'
df_ing['NIVELESAPROBADOS'] = df_ing['NIVELESAPROBADOS'].astype(int)

In [55]:
df_ing2 = pd.concat([df_ing,df_dm_conteo],ignore_index=True)

In [56]:
# Ordenar por NUM_IDENTIFICACION y luego por NIVELES_APROBADOS en orden descendente
df_ing2 = df_ing2.sort_values(by=['NUM_IDENTIFICACION', 'NIVELESAPROBADOS'], ascending=[True, False])

# Eliminar duplicados, manteniendo solo el registro con mayor número de niveles aprobados
df_ing2 = df_ing2.drop_duplicates(subset=['NUM_IDENTIFICACION'], keep='first')

In [57]:
df_ing2[df_ing2['NUM_IDENTIFICACION']=='1000046903']

,NUM_IDENTIFICACION,NIVELESAPROBADOS,TIPO_INGLES_MALLA
44589,1000046903,10,FUERA DE MALLA


In [58]:
result = df_op.merge(df_ing2[['NUM_IDENTIFICACION','NIVELESAPROBADOS','TIPO_INGLES_MALLA']],left_on=['Identificacion'],right_on=['NUM_IDENTIFICACION'],how='left')

In [59]:
import numpy as np
result['INGLES'] = np.where(
    result['TIPO_INGLES_MALLA'] == 'FUERA DE MALLA',
    np.where(
        (result['ciclo'] == 'TECNICO') & (result['NIVELESAPROBADOS'] >= 3), 'APROBÓ',
        np.where(
            (result['ciclo'] == 'TECNOLOGO') & (result['NIVELESAPROBADOS'] >= 6), 'APROBÓ',
            np.where(
                (result['ciclo'] == 'PROFESIONAL') & (result['NIVELESAPROBADOS'] >= 9), 'APROBÓ',
                'NO CUMPLE'  # Se mantiene 'NO CUMPLE' solo para los casos dentro de 'FUERA DE MALLA'
            )
        )
    ),
    ''
)

In [60]:
result['pensum'] = result['pensum'].str.rstrip('ND')

#### SE VALIDA DE LOS QUE NO CUMPLEN CUALES ESTAN EN PERIODOS ACTIVOS O EN CONVOCATORIA INGLES SE DEBE CORRER Periodos_ingles.ipynb

In [61]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            SELECT DISTINCT NUM_IDENTIFICACION, Periodo AS Periodo_conv, Estado,Niveles as Cursando_niveles  
            FROM SRUBIO.TABLA_CONVOCATORIA_INGLES
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_conv_act = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [62]:
result = pd.merge(result,df_conv_act[['NUM_IDENTIFICACION','Periodo_conv','Estado','Cursando_niveles']],how='left',left_on=['Identificacion'],right_on=['NUM_IDENTIFICACION'])

In [63]:
result.loc[(result['INGLES'].isin(['']))&(result['Estado'].notna()),'INGLES'] = result['Estado']

In [64]:
# Diccionario con las reglas por pensum y ciclo
requisitos_ingles = {
    '8014': {'TECNICO': 2, 'TECNOLOGO': 3, 'PROFESIONAL': 7},
    '3029': {'TECNICO': 2, 'TECNOLOGO': 4, 'PROFESIONAL': 5},
    'VG001': {'TECNICO': 1, 'TECNOLOGO': 2, 'PROFESIONAL': 5},
    'default': {'TECNICO': 3, 'TECNOLOGO': 6, 'PROFESIONAL': 9}  # Reglas generales
}

# Función para evaluar la aprobación
def evaluar_aprobacion(row):
    if row['TIPO_INGLES_MALLA'] != 'INGLES INCLUIDO EN MALLA':
        return row.get('INGLES', '')  # Mantiene la marca actual si ya tiene un valor
    
    # Obtiene las reglas según el pensum, usa 'default' si no está en el diccionario
    reglas = requisitos_ingles.get(row['pensum'], requisitos_ingles['default'])
    
    # Evalúa si cumple con el mínimo de niveles requeridos
    return 'APROBÓ' if row['NIVELESAPROBADOS'] >= reglas.get(row['ciclo'], 0) else 'NO CUMPLE'


# Aplica la función a cada fila del DataFrame
result['INGLES'] = result.apply(evaluar_aprobacion, axis=1)

In [65]:
result.loc[result['pensum'].isin(['8014','3029','VG001']) & (result['NIVELESAPROBADOS'].isna()),'INGLES'] = 'NO CUMPLE'
result.loc[result['TIPO_INGLES_MALLA'].isna(),'TIPO_INGLES_MALLA'] = 'FUERA DE MALLA'

In [66]:
result.loc[(result['NIVELESAPROBADOS'].isna())&(result['INGLES']==''),'INGLES'] = 'NO CUMPLE'

In [67]:
# 1. Mapeo para ordenar los ciclos jerárquicamente
orden_ciclos = {'TECNICO': 1, 'TECNOLOGO': 2, 'PROFESIONAL': 3}
result['ORDEN_CICLO'] = result['ciclo'].map(orden_ciclos)

# 2. Convertir 'INGLES' a binario para saber quién aprobó
result['INGLES_BIN'] = result['INGLES'].map({'APROBÓ': 1, 'NO CUMPLE': 0})

# 3. Identificar el ciclo máximo aprobado por cada persona
max_ciclo_aprobado = result[result['INGLES_BIN'] == 1].groupby('Identificacion')['ORDEN_CICLO'].max().reset_index()
max_ciclo_aprobado.rename(columns={'ORDEN_CICLO': 'MAX_ORDEN_CICLO_APROBADO'}, inplace=True)

# 4. Unir al DataFrame original
result = result.merge(max_ciclo_aprobado, on='Identificacion', how='left')

# 5. Sobrescribir la columna INGLES según condición
result['INGLES'] = result.apply(
    lambda x: 'APROBÓ' if pd.notna(x['MAX_ORDEN_CICLO_APROBADO']) and x['ORDEN_CICLO'] <= x['MAX_ORDEN_CICLO_APROBADO'] 
    else x['INGLES'], axis=1
)

# 6. Eliminar columnas auxiliares
result = result.drop(columns=['ORDEN_CICLO', 'INGLES_BIN', 'MAX_ORDEN_CICLO_APROBADO'])

In [68]:
result.loc[(result['INGLES']=='NO CUMPLE')&(result['Estado'].notna()),'INGLES'] = result['Estado']

### PENSAMIENTO CUNISTA CORRER SCRIPT Homol_Pract_PC.ipynb

In [69]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text('''
                         
            
	SELECT DISTINCT * FROM SRUBIO.Est_pract_Pens_Cun_act
    
        ''')
       
        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_pract_pens = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

In [70]:
df_grado_opor = result.merge(
    df_pract_pens[['NUM_IDENTIFICACION','ciclo', 'PENSAMIENTO CUNISTA', 'PRACTICAS']],
    left_on=['Identificacion', 'ciclo'],
    right_on=['NUM_IDENTIFICACION', 'ciclo'],
    how='left'
)

In [71]:
df_grado_opor.loc[(df_grado_opor['PENSAMIENTO CUNISTA'].isna()),'PENSAMIENTO CUNISTA'] = 'SIN CURSAR'

df_grado_opor.loc[(df_grado_opor['PRACTICAS'].isna()),'PRACTICAS'] = 'SIN CURSAR'

In [72]:
df_grado_opor.loc[(df_grado_opor['ESTADO_ACADEMICO'].isna()),'ESTADO_ACADEMICO'] = 'NO APROBADO'

df_grado_opor.loc[(df_grado_opor['ESTADO_ACADEMICO']=='Aun no aprueba el 100% de materias'),'ESTADO_ACADEMICO'] = 'NO APROBADO'

In [73]:
# Aplicar la condición para Niveles_aprobados
df_grado_opor.loc[df_grado_opor['NIVELESAPROBADOS'] > 9, 'NIVELESAPROBADOS'] = 9

In [74]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'GRADO_OPORTUNO'
schema_name = 'SRUBIO'
# Send the data to sql server
df_grado_opor.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

### SE MAPEA SI EL ESTUDIANTE CUENTA CON SABER PRO

In [75]:
# with engine.connect() as connection:
#     try:
#         # 
#         sql_query = text('''
                         
#      SELECT DISTINCT 
#      (b.tipo_identificacion_btpr) AS TIPO_DOC_UNICO,
#      a.Identificacion AS NUMERO, 
#      b.fecha_nacimiento_constitucion_btpr AS FEC_NACIMIENTO
      
#      FROM SRUBIO.GRADO_OPORTUNO a
#      LEFT JOIN HUB_persona b
# 	    ON convert(varchar,a.[Identificacion]) = convert(varchar,b.identificacion_btpr)      
	

#         ''')
       
#         # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
#         df_saber = pd.read_sql_query(sql_query, engine)

#     finally:
        
#         connection.close()

In [76]:
# # Convertir la columna de fechas a formato datetime
# df_saber['FEC_NACIMIENTO'] = pd.to_datetime(df_saber['FEC_NACIMIENTO'])

In [77]:
# # Formatear la columna de fechas al formato dd/mm/aaaa
# df_saber['FEC_NACIMIENTO'] = df_saber['FEC_NACIMIENTO'].dt.strftime('%d/%m/%Y')

In [78]:
# def crear_body_auths(df):
#     body_auths = []
#     for index, row in df.iterrows():
#         body_auth = {
#             "tipoDocumento": row['TIPO_DOC_UNICO'],
#             "numeroDocumento": row['NUMERO'],
#             "fechaNacimiento": row['FEC_NACIMIENTO']    
#         }
#         body_auths.append(body_auth)
#         #print(body_auths)
#     return body_auths
        

# # Llamar a la función y obtener una lista de diccionarios
# lista_body_auths = crear_body_auths(df_saber)

In [79]:
# # URL que se toman desde postman
# url_auth = "https://resultadosbackendoci.icfes.edu.co/api/seguritypro/autenticacionResultadosSaberProYTyT"
# url_resultados = "https://resultadosbackendoci.icfes.edu.co/api/protyt/ProTyt/consultaGeneral"

In [80]:
# import requests
# from tqdm import tqdm

# def hacer_peticion_completa(url_auth, lista_body_auths, url_resultados):
#     resultados_finales = []
#     errores = []
    
#     for body_auth in tqdm(lista_body_auths, desc="Procesando autenticación y resultados"):
#         try:
#             # Primera petición: autenticación
#             response_auth = requests.post(url_auth, json=body_auth)
#             response_auth.raise_for_status()
            
#             # Extraer el token de autenticación
#             token = response_auth.json().get('token', '')
#             if not token:
#                 raise ValueError("Token no encontrado en la respuesta de autenticación")
            
#             # Procesar cada examen del estudiante (puede haber múltiples exámenes como Saber TyT, Saber Pro)
#             for autenticacion in response_auth.json().get('datosAutenticacion', []):
#                 datos_parametros = autenticacion.get('datosParametros', {})
#                 examen = datos_parametros.get('examen', 'Examen no encontrado')  # Tipo de examen (Saber TyT, Saber Pro, etc.)
#                 anio_examen = autenticacion.get('anioExamen', 'Año no encontrado')  # Año del examen
#                 periodo_examen = autenticacion.get('periodoExamen', 'Periodo no encontrado')  # Periodo del examen
#                 periodo_anio_examen = datos_parametros.get('periodoAnioExamen', 'Periodo Año no encontrado')  # Periodo y Año del examen
#                 numero_registro = autenticacion.get('numeroRegistro', 'Registro no encontrado')  # Número de registro para la segunda petición
                
#                 # Crear el cuerpo de la segunda petición con la estructura correcta
#                 registro_resultado = {
#                     'tipoDocumento': body_auth['tipoDocumento'],
#                     'numeroDocumento': body_auth['numeroDocumento'],
#                     'identificacionUnica': numero_registro  # Usamos el numeroRegistro del examen
#                 }
                
#                 # Segunda petición: resultados
#                 headers = {'Authorization': f'Bearer {token}'}
#                 response_resultados = requests.post(url_resultados, json=registro_resultado, headers=headers)
#                 response_resultados.raise_for_status()
                
#                 # Agregar el cuerpo de la petición, la respuesta, el tipo de examen, año, periodo y periodoAnio del examen a los resultados finales
#                 resultados_finales.append({
#                     'examen': examen,  # Tipo de examen (Saber Pro, Saber TyT, etc.)
#                     'anioExamen': anio_examen,  # Año del examen
#                     'periodoExamen': periodo_examen,  # Periodo del examen
#                     'periodoAnioExamen': periodo_anio_examen,  # Periodo y Año del examen
#                     'request': registro_resultado,  # Estructura de la petición enviada
#                     'response': response_resultados.json()  # Respuesta de la segunda petición
#                 })
                
#         except requests.exceptions.RequestException as e:
#             errores.append({"registro": body_auth, "error": str(e), "tipo": "HTTP"})
#         except ValueError as e:
#             errores.append({"registro": body_auth, "error": str(e), "tipo": "ValueError"})
#         except Exception as e:
#             errores.append({"registro": body_auth, "error": str(e), "tipo": "General"})
    
#     return resultados_finales, errores

# # Uso de la función combinada
# respuesta, errores = hacer_peticion_completa(url_auth, lista_body_auths, url_resultados)

In [81]:
# respuesta

In [82]:
# # Normalizar reporteIndividualesGenericas
# df_genericas = pd.json_normalize(
#     respuesta, 
#     record_path=['response', 'reporteIndividualesGenericas'], 
#     meta=['request', 'examen', 'anioExamen', 'periodoExamen', 'periodoAnioExamen', ['response', 'resultadosGenerales']]  # Incluimos examen, anioExamen, periodoExamen y periodoAnioExamen
# )

# # Normalizar reporteIndividualesEspecificas
# df_especificas = pd.json_normalize(
#     respuesta,
#     record_path=['response', 'reporteIndividualesEspecificas'], 
#     meta=['request', 'examen', 'anioExamen', 'periodoExamen', 'periodoAnioExamen', ['response', 'resultadosGenerales']]  # También incluimos examen, anioExamen, periodoExamen y periodoAnioExamen
# )

# # Agregar una columna 'tipo' para distinguir entre genéricas y específicas
# df_genericas['tipo'] = 'genericas'
# df_especificas['tipo'] = 'especificas'

# # Concatenar ambos DataFrames
# df = pd.concat([df_genericas, df_especificas], ignore_index=True)

In [83]:
# # Función para limpiar y convertir JSON a diccionario
# def update_df_with_json(df, json_col):
#     # Iterar sobre cada fila del DataFrame
#     for index, row in df.iterrows():
        
#         json_data = row[json_col]
        
#         # Iterar sobre las claves y valores del diccionario JSON
#         for key, value in json_data.items():
#             if key in df.columns:  # Verificar si la clave está en las columnas del DataFrame
#                 df.at[index, key] = value
#             else:  # Agregar nuevas columnas si no existen
#                 df[key] = None
#                 df.at[index, key] = value

#     return df

# # Suponiendo que df es tu DataFrame y 'request' es la columna con JSON
# df_final = update_df_with_json(df, 'request')
# df_final = update_df_with_json(df, 'response.resultadosGenerales')

In [84]:
# df_final

In [85]:
# df_final.rename(columns={"tituloPercentilNBC":"tituloPercentilNBC_2"}, inplace=True)
# df_final.rename(columns={"percentilNBC":"percentilNBC_2"}, inplace=True)

In [86]:
# df_final.to_excel('Pruebas_saberPro_grado_oportuno.xlsx',index=False)

In [87]:
# df_final2 = df_final.copy()

In [88]:
# # Función para convertir listas a cadenas separadas por comas
# def convertir_a_cadena(x):
#     if isinstance(x, list):
#         return ', '.join(map(str, x))
#     return x

In [89]:
# # Aplicar la función a la columna
# df_final2['descripcionHabilidades'] = df_final2['descripcionHabilidades'].apply(convertir_a_cadena)

In [90]:
# df_final3 = df_final2[['nombrePrueba', 'tituloPruebaNacional', 'percentilNacional',
#        'tituloPercentilNbc', 'percentilNbc', 'nivelDesempeno',
#        'tituloHabilidades', 'descripcionHabilidades', 'ordenPrueba',
#        'puntajePrueba', 'totalPuntajePrueba', 'codigoPrueba', 'tipoPrueba',
#        'identificadorNBCAlerta', 'identificadorGrupoEspecialAlerta',
#        'tituloPruebaGrupo', 'percentilGrupoEvaluado', 'tipo', 'tipoDocumento',
#        'numeroDocumento', 'identificacionUnica', 'puntajeGlobal',
#        'puntajeTotalGlobal', 'tituloPercentilNacional',
#        'descripcionPercentilNacional', 'tituloPercentilNBC_2',
#        'percentilNBC_2', 'descripcionPercentilNBC', 'nbc',
#        'tituloPercentilGrupo', 'percentilGrupo', 'descripcionPercentilGrupo','examen']]

In [91]:
# # Connection to database
# host = '172.16.1.33'
# port = '1433'
# database = 'CUN_REPOSITORIO'
# # username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# # password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
# #url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
# url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
# engine = create_engine(url, fast_executemany=True)

# table_name = 'SABER_PRO_GRADO_OPORTUNO'
# schema_name = 'SRUBIO'
# # Send the data to sql server
# df_final3.to_sql(
#     name=table_name,
#     schema=schema_name,
#     con=engine,
#     if_exists='replace',
#     index=False,
#     chunksize = 10000
# )

# # End connection
# engine.dispose()

#### EJECUTAR Ultimo_periodo.ipynb

# AQUI TERMINA LA EJECUCIÓN DE LOS NOTEBOOKS